# Needle-in-a-Haystack Test — All Providers

Inserts 3 secret codes at depths 10%, 50%, 75% of a large filler document,
then asks the model to recall all three. Tests that a large context window
is actually loaded and used.

In [25]:
import random
import subprocess
import time
from pathlib import Path

from unified_local_llm_server import LocalLLMServer
from unified_local_llm_server.provider_registry import ProviderRegistry

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
registry = ProviderRegistry.load(ROOT / "providers.example.yaml")
server = LocalLLMServer(provider_registry=registry)

def vram_used_mib() -> int:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            text=True, timeout=5,
        )
        return sum(int(x) for x in out.strip().split("\n") if x.strip())
    except Exception:
        return 0

## Configuration

In [26]:
NITH_CTX = 110_000

PREFERRED_MODELS = {
    "ollama":    "gpt-oss:20b",
    "lm_studio": "openai/gpt-oss-20b",
    "unsloth":   "unsloth/gpt-oss-20b-GGUF@UD-Q4_K_XL",
    "llama_cpp": "gpt-oss-20b-MXFP4",
}

NEEDLES = [
    {"depth": 0.10, "text": "The first secret code is: PINEAPPLE7742",  "key": "PINEAPPLE7742"},
    {"depth": 0.50, "text": "The second secret code is: MANGO3318",     "key": "MANGO3318"},
    {"depth": 0.75, "text": "The third secret code is: KIWI9051",       "key": "KIWI9051"},
]

NEEDLE_QUERY = (
    "There are exactly three secret codes hidden in the document above. "
    "List all three codes, one per line. Reply with ONLY the codes, nothing else."
)

_FILLER_SENTENCES = [
    "The annual rainfall in the central region averages about 482 millimeters per year.",
    "Municipal water treatment plants process approximately 35 million liters daily.",
    "Crop rotation schedules typically follow a three-year cycle of cereals and legumes.",
    "Regional transportation budgets are allocated based on population density metrics.",
    "Standardised testing protocols require calibration of instruments every 90 days.",
    "Warehouse inventory systems use barcode scanning for tracking inbound shipments.",
    "Public library cataloguing follows the Dewey Decimal Classification system.",
    "Meteorological stations record wind speed, humidity, and barometric pressure hourly.",
    "Urban planning guidelines recommend a minimum of 12 square meters of green space per resident.",
    "Quality assurance audits are conducted on a quarterly basis across all manufacturing lines.",
    "The historical archive contains over 1.2 million digitized documents from the 19th century.",
    "Soil composition analysis requires pH testing alongside nitrogen and phosphorus measurements.",
    "The regional power grid distributes energy from 14 substations across the district.",
    "Building codes mandate seismic resistance ratings for structures above three stories.",
    "Statistical sampling methods use a confidence interval of 95 percent for survey data.",
]


def build_haystack(target_tokens: int, chars_per_token: float = 4.0) -> str:
    target_chars = int(target_tokens * chars_per_token)
    parts: list[str] = []
    total = 0
    while total < target_chars:
        s = random.choice(_FILLER_SENTENCES)
        parts.append(s)
        total += len(s) + 1
    filler = " ".join(parts)[:target_chars]
    for needle in sorted(NEEDLES, key=lambda n: n["depth"], reverse=True):
        pos = int(len(filler) * needle["depth"])
        pos = filler.rfind(". ", 0, pos + 1)
        pos = (pos + 2) if pos != -1 else 0
        filler = filler[:pos] + "\n\n" + needle["text"] + "\n\n" + filler[pos:]
    return filler


def check_needles(response: str) -> dict[str, bool]:
    normalized = response.upper().replace(" ", "")
    return {n["key"]: n["key"] in normalized for n in NEEDLES}


print(f"Building haystack for {NITH_CTX:,} tokens...")
haystack = build_haystack(NITH_CTX)
print(f"Done: {len(haystack):,} chars")

Building haystack for 110,000 tokens...
Done: 440,121 chars


## Provider Status

In [27]:
statuses = {}
for name in server.get_providers():
    statuses[name] = await server.check_provider_by_name(name)
    ok = statuses[name]["ok"]
    url = statuses[name]["server_url"]
    print(f"  {'OK' if ok else '--'} {name:<12} {url}")

  OK llama_cpp    http://127.0.0.1:9090
  OK lm_studio    http://127.0.0.1:1234
  OK ollama       http://127.0.0.1:11434
  OK unsloth      http://127.0.0.1:8899


## Run NITH Test

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant. Read the following document carefully, then answer the question.",
    },
    {
        "role": "user",
        "content": haystack + "\n\n" + NEEDLE_QUERY,
    },
]

nith_results = []

for provider in server.get_providers():
    if not statuses[provider]["ok"]:
        print(f"{provider} SKIP")
        continue
    model = PREFERRED_MODELS.get(provider)
    if not model:
        continue

    print()
    print("=" * 60)
    print(f"  {provider} / {model}")
    print("=" * 60)

    try:
        server.unload_all_models(provider)
        time.sleep(2)
    except Exception:
        pass

    vram_before = vram_used_mib()

    try:
        llm = server.load_model(provider, model, context_length=NITH_CTX)
        t0 = time.perf_counter()
        result, usage = await llm.call(
            return_usage=True,
            messages=messages,
            temperature=0.0,
            options={"num_predict": 200},
        )
        elapsed = time.perf_counter() - t0
    except Exception as exc:
        print(f"  ERROR: {exc}")
        nith_results.append({"provider": provider, "model": model, "error": str(exc)})
        continue

    vram_after = vram_used_mib()
    vram_delta = vram_after - vram_before

    prompt_tok = usage.get("prompt_tokens", 0)
    gen_tok    = usage.get("completion_tokens", 0)
    total_tok  = prompt_tok + gen_tok
    gen_tps    = round(gen_tok / elapsed, 1) if elapsed > 0 else 0

    found    = check_needles(result)
    all_pass = all(found.values())
    status   = "ALL PASS" if all_pass else ("PARTIAL" if any(found.values()) else "ALL FAIL")

    print(f"  Status       : {status}")
    for key, ok in found.items():
        print(f"    {key}: {'PASS' if ok else 'FAIL'}")
    print(f"  Prompt tokens: {prompt_tok}")
    print(f"  Gen tokens   : {gen_tok}")
    print(f"  Total tokens : {total_tok}")
    print(f"  Gen speed    : {gen_tps} tok/s")
    print(f"  Total time   : {round(elapsed, 1)} s")
    print(f"  VRAM delta   : {vram_delta} MiB")
    print(f"  Reply ({len(result)} chars):")
    print(result)

    nith_results.append({
        "provider":        provider,
        "model":           model,
        "ctx":             NITH_CTX,
        "found":           found,
        "all_pass":        all_pass,
        "status":          status,
        "prompt_tokens":   prompt_tok,
        "gen_tokens":      gen_tok,
        "total_tokens":    total_tok,
        "gen_tps":         gen_tps,
        "elapsed_s":       round(elapsed, 1),
        "vram_before_mib": vram_before,
        "vram_after_mib":  vram_after,
        "vram_delta_mib":  vram_delta,
        "response":        result,
    })

    try:
        server.unload_all_models(provider)
        time.sleep(1)
    except Exception:
        pass

## Summary

In [29]:
needle_keys = [n["key"] for n in NEEDLES]
col_w = 14
header = "{:<12}  {:>8}  {:>12}  {:>10}  {:>10}  {:>8}  {:>8}  {:>10}".format(
    "Provider", "ctx", "Status", "Time(s)", "Total tok", "P.tok", "G.tok/s", "VRAM MiB"
)
for k in needle_keys:
    header += f"  {k:>{col_w}}"
print(header)
print("-" * (len(header) + 2))

for r in nith_results:
    if "error" in r:
        print("{:<12}  {:>8}  ERROR: {}".format(r["provider"], NITH_CTX, r["error"][:60]))
        continue
    f = r["found"]
    row = "{:<12}  {:>8}  {:>12}  {:>10.1f}  {:>10}  {:>8}  {:>8.1f}  {:>10}".format(
        r["provider"],
        r["ctx"],
        r["status"],
        r["elapsed_s"],
        r["total_tokens"],
        r["prompt_tokens"],
        r["gen_tps"],
        r["vram_delta_mib"],
    )
    for k in needle_keys:
        row += f"  {'PASS' if f.get(k) else 'FAIL':>{col_w}}"
    print(row)

Provider           ctx        Status     Time(s)   Total tok     P.tok   G.tok/s    VRAM MiB   PINEAPPLE7742       MANGO3318        KIWI9051
----------------------------------------------------------------------------------------------------------------------------------------------
unsloth         110000      ALL PASS        31.6       72195     71997       6.3       14046            PASS            PASS            PASS
